# 1. Config

In [1]:
import pandas as pd
import openpyxl
import re
import numpy as np
import ftfy
from config import RAW_DATA_DIR, PROCESSED_DATA_DIR, FIGURES_DIR

import pandas as pd
from plotnine import *

pd.set_option('future.no_silent_downcasting', True)

# 2. Read data

In [2]:
file_path = RAW_DATA_DIR / "FCTAES_279691_TOTAL_20251006_ai_columns.xlsx"

data_df = pd.read_excel(file_path, sheet_name="Datos")
labels_df = pd.read_excel(file_path, sheet_name="Labels")
variables_df = pd.read_excel(file_path, sheet_name="Variables")
codes_df = pd.read_excel(file_path, sheet_name="Codes")

print("Datos",data_df.shape)
print("Labels",labels_df.shape)
print("Variables",variables_df.shape)
print("Codes",codes_df.shape)

Datos (1813, 514)
Labels (1813, 514)
Variables (515, 6)
Codes (2442, 3)


# 3. Data exploration & Character Cleanup

## Data exploration

In [3]:
# Check if metadata matches data columns
missing_defs = set(data_df.columns) - set(variables_df['Variable'])
print(f"Variables in data without definitions: {len(missing_defs)}")

# Quick look at the first few rows of each
display(data_df.head(3))
display(variables_df.head(3))

Variables in data without definitions: 0


,key,numericalId,accessCount,startTime,endTime,duration,status,type,SAMPLE,CodPanelista,...,P84_cod3,P84_cod4,P84_cod5,P84_cod6,P91_cod1,P91_cod2,P91_cod3,P91_cod4,P91_cod5,P91_cod6
0,2fd1e655-a94d-4a91-a4fa-ed22b097ce62,684173892,1,2025-09-22 12:17:19,2025-09-22 12:32:20,900,end,complete,1,01519d71455139b9,...,NaN,NaN,NaN,NaN,15.0,NaN,NaN,NaN,NaN,NaN
1,7da18c0c-caa8-4c5e-a5de-47f0805bb06e,684295712,1,2025-09-23 06:00:19,2025-09-23 06:15:05,886,end,complete,1,02082c28ac6c270d,...,NaN,NaN,NaN,NaN,54.0,NaN,NaN,NaN,NaN,NaN
2,f4fe2017-372a-4d5b-8a7c-6604cc560ec9,684182100,2,2025-09-22 13:36:32,2025-09-23 08:55:05,1293,end,complete,1,0248c65078193632,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,Variable,Posición,Etiqueta,Nivel de medición,original_code,new_name
0,key,1.0,key,Nominal,key,key
1,numericalId,2.0,numericalId,Escala,numericalId,numerical_id
2,accessCount,3.0,accessCount,Nominal,accessCount,access_count


In [4]:
# Calculate null percentages
null_report = data_df.isnull().mean() * 100
print(f"Percentage of columns with > 10% missing: {100 * (null_report > 10).sum() / len(null_report):.1f}%")

# Check for duplicates (crucial before merging or grouping)
print(f"Duplicate rows in data: {data_df.duplicated().sum()}")

Percentage of columns with > 10% missing: 94.6%
Duplicate rows in data: 0


In [5]:
data_df.columns[:100]

Index(['key', 'numericalId', 'accessCount', 'startTime', 'endTime', 'duration',
       'status', 'type', 'SAMPLE', 'CodPanelista', 'ppi_tp', 'SAMPLE_PROVIDER',
       'SURVEY_COUNTRY', 'DEVICE', 'SEXO', 'EDAD', 'EDADR', 'CITY_ES',
       'CITY_IT', 'CITY_GB', 'CITY_DK', 'CITY_GR', 'P2#1', 'P2#2', 'P2#3',
       'P2#97', 'TRANSPORT_1', 'TRANSPORT_2', 'TRANSPORT_3', 'P3#1', 'P3#2',
       'P3#3', 'P3#4', 'P4', 'P5', 'P6#1', 'P6#2', 'P6#96', 'P7_1', 'P7_2',
       'P7_3', 'P7_4', 'P7_5', 'P7_6', 'P8#1', 'P8#2', 'P8#3', 'P8#97', 'P9#1',
       'P9#2', 'P9#3', 'P9#4', 'P9#5', 'P9#6', 'P9#7', 'P9#8', 'P9#9', 'P9#10',
       'P9#11', 'P9#12', 'P9#96', 'P9#96#value', 'P10_1', 'P10_2', 'P10_3',
       'P10_4', 'P11_1', 'P11_2', 'P11_3', 'P11_4', 'P12_1', 'P12_2', 'P12_3',
       'P12_4', 'P13', 'P14#1', 'P14#2', 'P14#3', 'P14#4', 'P14#5', 'P14#6',
       'P14#7', 'P14#8', 'P14#9', 'P14#10', 'P14#11', 'P14#96', 'P14#96#value',
       'P15_1', 'P15_2', 'P15_3', 'P15_4', 'P15_5', 'P15_6', 'P15_7',

In [6]:
# Check unique values for the target columns you plan to invert
target_cols = ['P13', 'P17'] # Replace with your actual names
for col in target_cols:
    print(f"Value counts for {col}:\n", labels_df[col].value_counts().sort_index())

cities_cols = ['CITY_ES','CITY_IT', 'CITY_GB', 'CITY_DK', 'CITY_GR',]
# Check unique cities before the 'Atenas' transformation
print("Unique cities found:", labels_df[cities_cols].stack().unique())

Value counts for P13:
 P13
Always        201
Most times    259
Never           2
Sometimes      84
Name: count, dtype: int64
Value counts for P17:
 P17
Always        128
Most times    108
Never           9
Sometimes      69
Name: count, dtype: int64
Unique cities found: ['Barcelona' 'Roma' 'Copenhagen' 'London' 'AthÃ\xadna']


## Character Cleanup

In [7]:
# Store original values to detect changes
original_data_df = data_df.copy()
original_labels_df = labels_df.copy()
original_variables_df = variables_df.copy()
original_codes_df = codes_df.copy()

# Fix broken characters
data_df = data_df.map(lambda x: ftfy.fix_text(str(x)) if isinstance(x, str) else x)
labels_df = labels_df.map(lambda x: ftfy.fix_text(str(x)) if isinstance(x, str) else x)
variables_df = variables_df.map(lambda x: ftfy.fix_text(str(x)) if isinstance(x, str) else x)
codes_df = codes_df.map(lambda x: ftfy.fix_text(str(x)) if isinstance(x, str) else x)

# Print changes summary ignoring NaN values
print("Changes detected:")
print(f"data_df: {data_df.compare(original_data_df).size // 2} cells changed")
print(f"labels_df: {labels_df.compare(original_labels_df).size // 2} cells changed")
print(f"variables_df: {variables_df.compare(original_variables_df).size // 2} cells changed")
print(f"codes_df: {codes_df.compare(original_codes_df).size // 2} cells changed")

Changes detected:
data_df: 741 cells changed
labels_df: 5138 cells changed
variables_df: 121 cells changed
codes_df: 8 cells changed


## Metadata refinement

In [8]:
# --- Refine codes_df ---
# Ensure columns are named correctly before cleaning
codes_df.columns = ["Variable", "Code", "Label"]

# Remove the header row if it's redundant (semantic check)
codes_df = codes_df[codes_df["Variable"] != "Variable"].reset_index(drop=True)

# PROPAGATE VARIABLE NAMES: Fills the NaN values with the last valid variable name
codes_df['Variable'] = codes_df['Variable'].ffill()



# --- Refine variables_df ---
# The last row contains "Variables en el archivo de trabajo" 
variables_df = variables_df.iloc[:514].reset_index(drop=True)

print(f"Metadata Cleaned: Codes {codes_df.shape}, Variables {variables_df.shape}")



Metadata Cleaned: Codes (2442, 3), Variables (514, 6)


# 4.Column Renaming (Homogenization) 

In [9]:
def final_survey_cleanup(df):

    def apply_section_logic(row):
        code = str(row['original_code'])
        name = str(row['new_name'])
        
        # Extract the question number as an integer for range checking
        q_num_match = re.search(r'P(\d+)', code)
        q_num = int(q_num_match.group(1)) if q_num_match else None

        # --- Section Logic ---
        if q_num is None or q_num <=2: # Screening/System variables
            prefix = "scr_"
        elif 3 <= q_num <= 20: # Section 2: Motorcyclists
            prefix = "mc_"
        elif 21 <= q_num <= 34: # Section 3: Cyclists
            prefix = "bk_"
        elif 35 <= q_num <= 46: # Section 4: E-scooters
            prefix = "es_"
        elif 47 <= q_num <= 61: # Section 5: PPE (Bike + Scooter)
            prefix = "ppe_vru_"
        elif 62 <= q_num <= 92: # Section 6: PPE (Motorbike)
            prefix = "mc_ppe_"
        elif q_num >= 94: # Section 7: Demographics
            prefix = "dem_"
        else:
            prefix = ""

        # Apply prefix if not already there
        if not name.startswith(prefix):
            name = prefix + name

        # Standardize "Reason" naming for clarity
        if any(x in name for x in ['no_gear', 'no_jacket', 'no_helmet']):
            if 'reason_' not in name:
                name = name.replace('no_', 'reason_no_')

        # Clean up qualitative themes
        name = name.replace('_cod', '_theme').replace('_sit', '').replace('_expl', '')

        return name.lower()

    df['new_name_final'] = df.apply(apply_section_logic, axis=1)

    return df

# Execute
variables_df_refined = final_survey_cleanup(variables_df)


In [10]:
variables_df_refined.head(40)

,Variable,Posición,Etiqueta,Nivel de medición,original_code,new_name,new_name_final
0,key,1.0,key,Nominal,key,key,scr_key
1,numericalId,2.0,numericalId,Escala,numericalId,numerical_id,scr_numerical_id
2,accessCount,3.0,accessCount,Nominal,accessCount,access_count,scr_access_count
3,startTime,4.0,startTime,Escala,startTime,start_time,scr_start_time
4,endTime,5.0,endTime,Escala,endTime,end_time,scr_end_time
5,duration,6.0,duration,Escala,duration,duration,scr_duration
6,status,7.0,status,Nominal,status,status,scr_status
7,type,8.0,type,Nominal,type,type,scr_type
8,SAMPLE,9.0,SAMPLE,Nominal,SAMPLE,sample,scr_sample
9,CodPanelista,10.0,CodPanelista,Nominal,CodPanelista,cod_panelista,scr_theme_panelista


# 5. Data Transformations & Feature Engineering

In [11]:
# Final dataframe 
final_df = data_df.copy()

## A. Geographic Variables

In [12]:
city_cols = ['CITY_ES', 'CITY_IT', 'CITY_GB', 'CITY_DK', 'CITY_GR']
final_df['City'] = (
    labels_df[city_cols]
    .stack()
    .reset_index(level=1, drop=True)
)

# Replace 'Athína' with 'Athens' in the City column
final_df['City'] = final_df['City'].replace('Athína', 'Athens')

print(final_df[['City']].value_counts())


City      
Barcelona     403
Roma          400
Copenhagen    400
Athens        310
London        300
Name: count, dtype: int64


## B. Age groups

In [13]:
# 1. Define Bins and Labels
age_bins = [0, 17, 24, 34, 44, 54, 64, 74, np.inf]
age_labels = ['<18', '18-24', '25-34', '35-44', '45-54', '55-64', '65-74', '75+']

# 2. Create the Categorical Labels
final_df["age_group"] = pd.cut(final_df["EDAD"], bins=age_bins, labels=age_labels)

# 3. Create the Ordinal Encoding (Elegant way: labels=False returns the bin index)
final_df["age_group_encoded"] = pd.cut(final_df["EDAD"], bins=age_bins, labels=False)

# Validation
print(final_df[["EDAD", "age_group", "age_group_encoded"]].head())
print("\nDistribution:\n", final_df["age_group"].value_counts().sort_index())

   EDAD age_group  age_group_encoded
0    35     35-44                  3
1    73     65-74                  6
2    42     35-44                  3
3    41     35-44                  3
4    31     25-34                  2

Distribution:
 age_group
<18        4
18-24     73
25-34    380
35-44    453
45-54    461
55-64    300
65-74    114
75+       28
Name: count, dtype: int64


## C. Change order usage

In [14]:
#Change the code in the original dataframe to match the new coding
final_df['P13_recoded'] = 5 - final_df['P13']
final_df['P17_recoded'] = 5 - final_df['P17']

#Change the code in the dictionary to match the new coding

# Reverse the codes for P13 and P17 in codes_df
# 1. Ensure the 'Code' column is actually numeric (handles string digits)
codes_df['Code'] = pd.to_numeric(codes_df['Code'], errors='coerce')

# 2. Define your mask
mask = codes_df['Variable'].isin(['P13', 'P17'])

# 3. Apply the transformation
# Using 5 - x assumes a 1-4 scale. Change 5 to (Max + Min) for other scales.
codes_df.loc[mask, 'Code'] = 5 - codes_df.loc[mask, 'Code']

# 4. Check the result


In [15]:
final_df['P7_3'].value_counts()

P7_3
1.0    681
2.0     69
4.0     53
3.0     52
5.0     50
Name: count, dtype: int64

In [16]:
# New Goal: 1 is 'Always', 4 is 'Never'
helmet_map_corrected = {
    5.0: 1.0,  # (Almost) Always -> Always (1)
    4.0: 2.0,  # Often -> Most times (2)
    3.0: 3.0,  # Sometimes -> Sometimes (3)
    2.0: 4.0,  # Seldom -> Never (4)
    1.0: 4.0   # Never -> Never (4)
}

# Apply to the dataframe
final_df['P7_3_recoded'] = final_df['P7_3'].map(helmet_map_corrected)

# Update the metadata to match this 1=Always logic
p7_metadata = pd.DataFrame({
    'Variable': ['P7_3_recoded'] * 4,
    'Code': [1.0, 2.0, 3.0, 4.0],
    'Label': ['Always', 'Most times', 'Sometimes', 'Never']
})

# Note: If you already ran the previous concat, you might want to 
# filter codes_df first to avoid duplicate P7_3_recoded entries.
codes_df = codes_df[codes_df['Variable'] != 'P7_3_recoded']
codes_df = pd.concat([codes_df, p7_metadata], ignore_index=True)

In [17]:
# Verify the alignment
codes_df[codes_df['Variable'].isin(['P13', 'P17', 'P7_3_recoded'])]


,Variable,Code,Label
271,P13,4.0,Always
272,P13,3.0,Most times
273,P13,2.0,Sometimes
274,P13,1.0,Never
353,P17,4.0,Always
354,P17,3.0,Most times
355,P17,2.0,Sometimes
356,P17,1.0,Never
2442,P7_3_recoded,1.0,Always
2443,P7_3_recoded,2.0,Most times


### D. Change statements

In [ ]:
# List of items to be reversed (The negative ones)
items_to_reverse = [4, 5, 6, 10]

# Recode P15 (Jacket) and P19 (Leg Protection)
for i in items_to_reverse:
    # Create the new recoded columns
    final_df[f'P15_{i}_recoded'] = 6 - final_df[f'P15_{i}']
    final_df[f'P19_{i}_recoded'] = 6 - final_df[f'P19_{i}']

# For the positive items, we just copy them to keep a consistent naming convention
items_to_keep = [1, 2, 3, 7, 8, 9]
for i in items_to_keep:
    final_df[f'P15_{i}_recoded'] = final_df[f'P15_{i}']
    final_df[f'P19_{i}_recoded'] = final_df[f'P19_{i}']

In [29]:
# Define the new standardized labels
new_labels = {
    1: "Carrying it is convenient",
    2: "Wearing it is pleasant",
    3: "Reduces risk of injury",
    4: "Necessary for short journeys",             # Reversed meaning
    5: "Necessary aside motorways",               # Reversed meaning
    6: "Necessary for all experience levels",      # Reversed meaning
    7: "Social Norm: Most people wear it",
    8: "Friends approve",
    9: "Family approve",
    10: "Authorities enforce wearing it"           # Reversed meaning
}

# Add these to your codes_df for both P15 and P19
metadata_list = []
for p_prefix in ['P15', 'P19']:
    for num, label in new_labels.items():
        for val in range(1, 6):
            metadata_list.append({
                'Variable': f'{p_prefix}_{num}',
                'Code': float(val),
                'Label': label # You could also add "Agree/Disagree" context here
            })

new_metadata_df = pd.DataFrame(metadata_list)
codes_df = pd.concat([codes_df, new_metadata_df], ignore_index=True)

In [31]:
codes_df[codes_df['Variable']=='P15_1']


,Variable,Code,Label
299,P15_1,1.0,1 Disagree
300,P15_1,2.0,2
301,P15_1,3.0,3
302,P15_1,4.0,4
303,P15_1,5.0,5 Agree
2546,P15_1,1.0,Carrying it is convenient
2547,P15_1,2.0,Carrying it is convenient
2548,P15_1,3.0,Carrying it is convenient
2549,P15_1,4.0,Carrying it is convenient
2550,P15_1,5.0,Carrying it is convenient


# Save data

In [21]:
variables_df_refined = variables_df_refined.rename(columns={"new_name_final": "coding_variable_name"})
variables_df_refined = variables_df_refined[['Variable', 'Posición', 'Etiqueta', 'Nivel de medición','coding_variable_name']]
variables_df_refined.columns

Index(['Variable', 'Posición', 'Etiqueta', 'Nivel de medición',
       'coding_variable_name'],
      dtype='object')

In [22]:
with pd.ExcelWriter(PROCESSED_DATA_DIR / "survey_processed_data.xlsx") as writer:
    final_df.to_excel(writer, sheet_name="Datos", index=False)
    labels_df.to_excel(writer, sheet_name="Labels", index=False)
    variables_df_refined.to_excel(writer, sheet_name="Variables", index=False)
    codes_df.to_excel(writer, sheet_name="Codes", index=False)
    
